In [3]:
!pip install japanize_matplotlib

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pickle
import matplotlib.animation as animation
import time
import japanize_matplotlib

from IPython.display import clear_output, display
from pylab import rcParams

rcParams['figure.figsize'] = 25, 20

# ============================================================
# Neural network settings
# ============================================================
i_nodes = 784
h_nodes = 300
o_nodes = 10

# ============================================================
# Load training data
# ============================================================
#f = pd.read_csv('mnist_train_10000.csv',  encoding='ms932',  sep=',',  header=None)

url = "https://github.com/ueharaLab/datascience_python8/raw/main/mnist_train_10000.csv"
f = pd.read_csv( url, header=None)

print(f.shape)
data = f.values

index = np.arange(len(f))
np.random.shuffle(index)

# ============================================================
# Number of training samples
# ============================================================
no_of_trains = int(input('何回学習しますか？'))

train_index = index[0:no_of_trains]

# IMPORTANT:
# Do not use the variable name "input",
# because it overwrites Python's built-in input() function.
input_data = np.asarray( data[train_index, 1:],  dtype=np.float64) / 255.0

# ============================================================
# Create one-hot target output
# ============================================================
correct_label = np.zeros( (no_of_trains, o_nodes))

for i, label in enumerate( data[train_index, 0].astype(int)):
    correct_label[i, label] = 1.0

# ============================================================
# Initialize weights
# ============================================================
w_z = (np.random.rand(h_nodes, o_nodes) - 0.51) / o_nodes
w_y = (np.random.rand(i_nodes, h_nodes) - 0.51) / h_nodes

learningRate = 0.1

# ============================================================
# Train neural network
# Display the current handwritten image every 100 iterations
# in the SAME Jupyter output area.
# ============================================================
for i, (x, t) in enumerate(zip(input_data, correct_label)):

    # --------------------------------------------------------
    # Jupyter visualization
    # --------------------------------------------------------
    if  i % 100 == 0:
        
        clear_output(wait=True)
        image_array = x.reshape((28, 28))
        fig, ax = plt.subplots(figsize=(6, 6))

        ax.imshow( image_array, cmap='Greys', interpolation='None'    )

        ax.axis("off")
        ax.set_title('Training count: ' + str(i)  + ' / '+ str(no_of_trains),  fontsize=26   )

        display(fig)
        plt.close(fig)

        time.sleep(0.3)

    # --------------------------------------------------------
    # Forward propagation
    # --------------------------------------------------------
    x = x.reshape(1, len(x))
    y_in = np.dot(x, w_y)
    y_out = 1 / (  1 + np.exp(-y_in)  )
    z_in = np.dot( y_out,  w_z )

    # Numerical stability for softmax
    #z_in = z_in - np.max( z_in, axis=1,  keepdims=True  )

    z_out = (  np.exp(z_in)  / np.sum( np.exp(z_in),  axis=1,  keepdims=True )  )

    # --------------------------------------------------------
    # Back propagation
    # --------------------------------------------------------
    delta_l = z_out - t
    grad_w_z = np.dot(  y_out.T,  delta_l  )
    delta_n = np.sum(delta_l * w_z,  axis=1 )
    delta_n = delta_n.reshape( 1, len(delta_n)  )
    delta_n = (  delta_n * (1 - y_out) * y_out  )

    # Alternative:
    # delta_n = np.dot(delta_l, w_z.T) * (1-y_out) * y_out

    grad_w_y = np.dot( x.T,  delta_n  )

    # --------------------------------------------------------
    # Gradient descent
    # --------------------------------------------------------
    w_z -= learningRate * grad_w_z
    w_y -= learningRate * grad_w_y

# ============================================================
# Training finished
# ============================================================
clear_output(wait=True)

fig, ax = plt.subplots(figsize=(10, 4))
ax.axis("off")
ax.text( 0.1,  0.5, 'データ' + str(no_of_trains)   + '件の学習が終わりました。',  size=30,  linespacing=2)
display(fig)
plt.close(fig)

time.sleep(2)

# ============================================================
# Load test data
# ============================================================
#f = pd.read_csv(  'mnist_test_10.csv',  encoding='ms932',  sep=',',  header=None)

url = "https://github.com/ueharaLab/datascience_python8/raw/main/mnist_test_10.csv"
f = pd.read_csv( url, header=None)
f = f.values
x = np.asarray(  f[:, 1:],   dtype=np.float64) / 255.0
correct_label_test = f[:, 0].astype(int)

# ============================================================
# Test
# ============================================================
y_in = np.dot( x, w_y)
y_out = 1 / (  1 + np.exp(-y_in))
z_in = np.dot(  y_out,  w_z)

# Numerical stability for softmax
z_in = z_in - np.max( z_in,  axis=1,    keepdims=True)

z_out = ( np.exp(z_in) /  np.sum(np.exp(z_in), axis=1, keepdims=True ))

# ============================================================
# Visualize 10 test results
# ============================================================
fig = plt.figure(figsize=(18, 8))

for i, (xx, zz, c) in enumerate( zip(x, z_out, correct_label_test)):

    image_array = np.asarray(xx).reshape((28, 28))
    ax = fig.add_subplot( 2, 5, i + 1 )
    ax.imshow( image_array, cmap='Greys',  interpolation='None'    )
    ax.axis("off")

    label = np.argmax(zz)

    ax.set_title( 'Correct Label: ' + str(c) + '\nPredict Value: ' + str(label),  fontsize=24  )

plt.tight_layout()
plt.show()


(10000, 785)


KeyboardInterrupt: Interrupted by user